# HTR baseline A: training

Stage 20–22 Oct (recognition). TrOCR fine-tuned on human labels only, ground-truth boxes, no augmentation.
Uses the split and crops written by `01_htr_pilot.ipynb`. The best epoch by greedy validation CER is saved
to `models/trocr-uk-baseline`; `03_htr_eval.ipynb` evaluates it. The official test split is not touched.

In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_linear_schedule_with_warmup
from transformers.utils import logging

sys.path.insert(0, '../src')
from htr import CROP_FILE, REVISION, LineDataset, cer, collate, predict, train_epoch, wer

logging.set_verbosity_error()
logging.disable_progress_bar()
pd.set_option('display.max_colwidth', 120)

CONFIG = dict(
    dataset_revision=REVISION,
    base_model='cyrillic-trocr/trocr-ukrainian-handwritten',
    base_revision='2f717bb4e44af7e5cabf553df2a9aae06322255e',
    split_file='data/splits/train_val.csv',
    seed=42,
    epochs=5,
    batch_size=16,
    lr=3e-5,
    weight_decay=0.01,
    warmup_fraction=0.1,
    max_target_tokens=128,
    selection_decoding=dict(num_beams=1, no_repeat_ngram_size=0, max_length=130),
)
MODEL_DIR = Path('../models/trocr-uk-baseline')
RESULTS_DIR = Path('../results/htr')
torch.manual_seed(CONFIG['seed'])

In [ ]:
processor = TrOCRProcessor.from_pretrained(CONFIG['base_model'], revision=CONFIG['base_revision'])
crops = pd.read_csv(CROP_FILE).dropna(subset=['target'])
too_long = crops.target.map(lambda t: len(processor.tokenizer(t).input_ids) > CONFIG['max_target_tokens'])
train = crops[(crops.split == 'train') & ~too_long]
val = crops[crops.split == 'val'].copy()
print(f'train: {len(train)} lines from {train.page_id.nunique()} pages ({too_long.sum()} over-long lines dropped)')
print(f'val:   {len(val)} lines from {val.page_id.nunique()} pages')

## Training

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(CONFIG['base_model'], revision=CONFIG['base_revision']).cuda()
model.gradient_checkpointing_enable()
loader = torch.utils.data.DataLoader(LineDataset(train, processor, CONFIG['max_target_tokens']),
                                     batch_size=CONFIG['batch_size'], shuffle=True, num_workers=4,
                                     collate_fn=lambda b: collate(b, processor.tokenizer.pad_token_id))
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'], fused=True)
total_steps = CONFIG['epochs'] * len(loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(CONFIG['warmup_fraction'] * total_steps), total_steps)

hyps = predict(model, processor, val, **CONFIG['selection_decoding'])
history = [dict(epoch=0, train_loss=None, val_CER=cer(val.target, hyps), val_WER=wer(val.target, hyps), minutes=0)]
for epoch in range(1, CONFIG['epochs'] + 1):
    start = time.time()
    losses = train_epoch(model, loader, optimizer, scheduler)
    hyps = predict(model, processor, val, **CONFIG['selection_decoding'])
    history.append(dict(epoch=epoch, train_loss=np.mean(losses), val_CER=cer(val.target, hyps),
                        val_WER=wer(val.target, hyps), minutes=(time.time() - start) / 60))
    if history[-1]['val_CER'] == min(h['val_CER'] for h in history[1:]):
        model.save_pretrained(MODEL_DIR)
        processor.save_pretrained(MODEL_DIR)
    print(history[-1])
history = pd.DataFrame(history)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'training.json').write_text(json.dumps(dict(
    config=CONFIG, history=history.replace({np.nan: None}).to_dict('records'),
), indent=2), encoding='utf-8')
history.round(4)

Epoch 0 is the Ukrainian checkpoint before fine-tuning. Config and history are saved to `results/htr/training.json`.